# Mushroom data preparation

**Contributors:** initial scaffold and local verification prepared with AI assistance. Team reviewers and substantive changes will be recorded after review. See [the assistance record](../docs/AI_USE.md).

Reproduce the development split and selected preprocessing without exploratory graphs. Categorical codes stay nominal. Numeric missing values use training medians and missingness indicators; missing categories receive a dedicated value before one-hot encoding.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src' / 'mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mushrooms import load_data, make_split, preprocessor
frame = load_data()
splits = make_split(frame)
X = frame.drop(columns='class')
y = frame['class'].map({'e': 0, 'p': 1})
X_train = X.loc[splits['train']]
X_validation = X.loc[splits['validation']]
y_train = y.loc[splits['train']]
y_validation = y.loc[splits['validation']]
print({name: len(indices) for name, indices in splits.items()})

{'train': 3000, 'validation': 1000, 'test': 1000}


## Fit preprocessing on training rows

The validation rows are transformed with the fitted training parameters. The reserved final test is not evaluated. In the modelling notebook these same steps stay inside each model pipeline; logistic regression additionally scales numeric features.

In [2]:
prepare = preprocessor(X_train)
prepared_train = prepare.fit_transform(X_train)
prepared_validation = prepare.transform(X_validation)
print('Prepared training shape:', prepared_train.shape)
print('Prepared validation shape:', prepared_validation.shape)
print('Positive class: p, encoded as 1. Final test remains reserved.')

Prepared training shape: (3000, 83)
Prepared validation shape: (1000, 83)
Positive class: p, encoded as 1. Final test remains reserved.


## Current preparation choices

No rows are removed merely because an input is missing. The noisy columns and highly missing columns remain in this baseline so their effect can be tested through controlled development experiments. Update this preparation when a justified cleaning change is selected, and keep the training script consistent with it.